In [ ]:
import caiman as cm
from caiman.utils.visualization import inspect_correlation_pnr, nb_inspect_correlation_pnr
import numpy as np
import gc
from ipywidgets import FloatSlider, interactive, IntSlider
import holoviews as hv
hv.extension('bokeh') 

### before caiman preproc

In [ ]:
# --- Load movie ---
m = cm.load('/media/toor/Seagate2/CaimanTesting/2024-06-10T17_53_09/TMaze_miniscope2024-06-10T17_53_09.avi')

# --- Randomly sample frames to reduce memory load ---
n_frames_total = m.shape[0]
n_frames_sample = min(3000, n_frames_total)  # use up to 3000 frames


In [ ]:

# pick frames evenly across the movie (not just first segment)
idx = np.linspace(0, n_frames_total - 1, n_frames_sample, dtype=int)
m_sample = m[idx]


In [ ]:
cn, pnr = cm.summary_images.correlation_pnr(
    m_sample, 
    gSig=4.5,          # estimated neuron half-size in pixels (adjust if needed)
    center_psf=True, # for miniscope data
    swap_dim=False
)


In [ ]:
# --- Interactive threshold tuning ---
nb_inspect_correlation_pnr(cn, pnr)


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# --- thresholds ---
min_corr = 0.65
min_pnr  = 5.5

# --- binary mask of candidate pixels ---
mask = (cn > min_corr) & (pnr > min_pnr)

# --- combined visualization ---
plt.figure(figsize=(10, 5))

im = plt.imshow(cn, cmap='gray')           # store handle to the image
plt.contour(mask, colors='red', linewidths=0.5)
plt.title(f'Candidate seeds (min_corr={min_corr}, min_pnr={min_pnr})')
plt.axis('off')
plt.colorbar(im, label='Correlation')      # use the handle here
plt.show()

# --- optional: histogram of correlation & pnr values for diagnostics ---
plt.figure(figsize=(8,3))
plt.subplot(1,2,1)
plt.hist(cn.flatten(), bins=100, color='gray')
plt.axvline(min_corr, color='r', linestyle='--')
plt.title('Correlation distribution')

plt.subplot(1,2,2)
plt.hist(pnr.flatten(), bins=100, color='gray')
plt.axvline(min_pnr, color='r', linestyle='--')
plt.title('PNR distribution')
plt.tight_layout()
plt.show()


### parameter evaluation

In [ ]:
from caiman.source_extraction.cnmf.cnmf import load_CNMF

# Replace with your path
cnmf_path1 = '/media/toor/Seagate2/CaimanTesting/2024-06-13T15_32_54/caiman/caiman_results.hdf5'
cnmf_path2 = '/media/toor/Seagate2/CaimanTesting/2024-06-13T15_32_54/caiman2/caiman_results.hdf5'
cnmf_path3 = '/media/toor/Seagate2/Mouse847/Round1/Linear/2025-11-10T17_24_58/caiman_final/caiman_results.hdf5'
cnmf_path4 = '/media/toor/Seagate2/CaimanTesting/2024-06-13T15_32_54/caiman4/caiman_results.hdf5'
cnmf_path5 = '/media/toor/Seagate2/CaimanTesting/2024-06-13T15_32_54/caiman5/caiman_results.hdf5'
cnmf_path_final = '/media/toor/Seagate2/CaimanTesting/2024-06-10T17_53_09/caiman_final/caiman_results.hdf5'
cnmf_newData = '/media/toor/Seagate2/Mouse847/caiman_55_denoised4/caiman_results.hdf5'
cnm=load_CNMF(cnmf_path3)


In [ ]:
# Get dimensions
dims = cnm.params.data['dims']  # e.g., (480, 640)
A = cnm.estimates.A.toarray().reshape((*dims, -1), order='F')  # reshape from 1D to 2D maps


In [ ]:
import matplotlib.pyplot as plt
cn_img = cnm.estimates.Cn 

plt.figure(figsize=(10, 6)) 
plt.imshow(cn_img, cmap='gray') 

# Overlay the accepted components 
for i in cnm.estimates.idx_components: 
    footprint = A[:, :, i] 
    plt.contour(footprint, levels=[0.2 * A[:, :, i].max()], colors='red', linewidths=0.5) 
    
# Compute centroid of the component 
    y, x = np.where(footprint > 0.5 * footprint.max()) 
    if len(x) > 0: 
        x_center, y_center = np.mean(x), np.mean(y) 
        plt.text(x_center, y_center, str(i), color='r', fontsize=6, weight='bold', ha='center', va='center') 
plt.title(f'Spatial footprints of accepted components (n={len(cnm.estimates.idx_components)})') 
plt.axis('off') 
plt.show()

In [ ]:
cnm.estimates.nb_view_components(img=cn_img,
                                 idx=cnm.estimates.idx_components[::5],
                                 cmap='gray',
                                 denoised_color='red'
                                 )

In [ ]:
cnm.estimates.nb_view_components(img=cn_img,
                                 idx=cnm.estimates.idx_components_bad[::5],
                                 cmap='gray',
                                 denoised_color='red'
                                 )

In [ ]:
import numpy as np

i = 5

q = cnm.params.quality
snr = float(cnm.estimates.SNR_comp[i])
rval = float(cnm.estimates.r_values[i])
cnn  = float(cnm.estimates.cnn_preds[i]) if hasattr(cnm.estimates, 'cnn_preds') else np.nan

reasons = []
if snr < q['SNR_lowest'] or snr < q['min_SNR']:
    reasons.append(f"SNR {snr:.2f} < min {q['min_SNR']:.2f}")
if rval < q['rval_lowest'] or rval < q['rval_thr']:
    reasons.append(f"r_value {rval:.3f} < min {q['rval_thr']:.2f}")
if q.get('use_cnn', False):
    if cnn < q['cnn_lowest'] or cnn < q['min_cnn_thr']:
        reasons.append(f"CNN {cnn:.3f} < min {q['min_cnn_thr']:.2f}")

print(f"Cell {i} SNR={snr:.2f}, r={rval:.3f}, CNN={cnn:.3f}  ->  reasons: {reasons or 'ACCEPTED'}")


### Alter evaluation params in place -  doesn't work

In [ ]:
# Soften or disable CNN gating
cnm.params.change_params({
    'quality': {
        'use_cnn': True,        # or False to disable entirely
        'min_cnn_thr': 0.9,    # accept anything ≥ 0.85 instead of 0.99
        'cnn_lowest': 0.10      # allow low scores without auto-rejection
    }
})

# Re-apply selection with the new criteria
# cnm.estimates.evaluate_components(cnm.estimates.Cn, cnm.params, dview=None)
cnm.estimates.select_components(use_object=True)


In [ ]:
cn_img = cnm.estimates.Cn 

plt.figure(figsize=(10, 6)) 
plt.imshow(cn_img, cmap='gray') 

# Overlay the accepted components 
for i in cnm.estimates.idx_components: 
    footprint = A[:, :, i] 
    plt.contour(footprint, levels=[0.2 * A[:, :, i].max()], colors='red', linewidths=0.5) 
    
# Compute centroid of the component 
    y, x = np.where(footprint > 0.5 * footprint.max()) 
    if len(x) > 0: 
        x_center, y_center = np.mean(x), np.mean(y) 
        plt.text(x_center, y_center, str(i), color='r', fontsize=6, weight='bold', ha='center', va='center') 
plt.title(f'Spatial footprints of accepted components (n={len(cnm.estimates.idx_components)})') 
plt.axis('off') 
plt.show()

### Population Evaluation

In [ ]:
from caiman.source_extraction.cnmf.cnmf import load_CNMF

# Replace with your path
cnmf_path1 = '/media/toor/Seagate2/CaimanTesting/2024-06-13T15_32_54/caiman/caiman_results.hdf5'
cnmf_path2 = '/media/toor/Seagate2/CaimanTesting/2024-06-13T15_32_54/caiman2/caiman_results.hdf5'
cnmf_path3 = '/media/toor/Seagate2/CaimanTesting/2024-06-13T15_32_54/caiman3/caiman_results.hdf5'
cnmf_path4 = '/media/toor/Seagate2/CaimanTesting/2024-06-10T17_53_09/caiman4/caiman_results.hdf5'
cnmf_path5 = '/media/toor/Seagate2/CaimanTesting/2024-06-10T17_53_09/caiman_55/caiman_results.hdf5'
cnmf_path_final = '/media/toor/Seagate2/CaimanTesting/2024-06-10T17_53_09/caiman_final/caiman_results.hdf5'
cnmf_path_biblio = '/media/toor/Seagate2/CaimanTesting/2024-06-10T17_53_09/caiman_biblio/caiman_results.hdf5'
cnmf_path_biblio_contoursize = '/media/toor/Seagate2/CaimanTesting/2024-06-10T17_53_09/caiman_biblio_contoursize/caiman_results.hdf5'
cnm=load_CNMF(cnmf_path_biblio_contoursize)
cnm1 = load_CNMF(cnmf_path5)
cnm2 = load_CNMF(cnmf_path_biblio)


In [ ]:
import pandas as pd
import numpy as np

def summarize_cnmf(cnm, label):
    """Return summary metrics for one CNMF model."""
    est = cnm.estimates
    n_total = est.A.shape[1]
    n_good = len(est.idx_components)
    n_bad = len(est.idx_components_bad)

    snr = np.asarray(est.SNR_comp)
    rval = np.asarray(est.r_values)
    cnn = np.asarray(getattr(est, 'cnn_preds', np.full_like(snr, np.nan)))

    return {
        'Model': label,
        'Accepted': n_good,
        'Rejected': n_bad,
        'Total': n_total,
        'Mean SNR': np.nanmean(snr),
        'Median SNR': np.nanmedian(snr),
        'Mean r_value': np.nanmean(rval),
        'Median r_value': np.nanmedian(rval),
        'Mean CNN': np.nanmean(cnn),
        'Median CNN': np.nanmedian(cnn)
    }

# Example: summarize multiple models
summaries = []
summaries.append(summarize_cnmf(cnm2, 'biblio1'))
summaries.append(summarize_cnmf(cnm1, 'gSig=5'))
summaries.append(summarize_cnmf(cnm, 'contour_size'))

df_summary = pd.DataFrame(summaries)
display(df_summary)


In [ ]:
import matplotlib.pyplot as plt

def qc_scatter(cnm, title):
    est = cnm.estimates
    snr = np.asarray(est.SNR_comp)
    rval = np.asarray(est.r_values)
    cnn = np.asarray(getattr(est, 'cnn_preds', np.full_like(snr, np.nan)))

    good_mask = np.isin(range(len(snr)), est.idx_components)
    bad_mask = np.isin(range(len(snr)), est.idx_components_bad)

    plt.figure(figsize=(6,5))
    plt.scatter(rval[bad_mask], snr[bad_mask], color='gray', alpha=0.5, label='Rejected')
    plt.scatter(rval[good_mask], snr[good_mask], color='green', alpha=0.7, label='Accepted')
    plt.xlabel('Spatial correlation (r_value)')
    plt.ylabel('SNR')
    plt.title(f'{title}\nAccepted={good_mask.sum()}, Rejected={bad_mask.sum()}')
    plt.legend()
    plt.grid(True)
    plt.show()

# Example for one model
qc_scatter(cnm, 'Model: contour_size')
qc_scatter(cnm1, 'Model: gSig=5')
qc_scatter(cnm2, 'Model: biblio')


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

def summarize_cnmf(cnm, label):
    est = cnm.estimates
    n_total = est.A.shape[1]
    n_good = len(est.idx_components)
    n_bad = len(est.idx_components_bad)

    snr = np.asarray(est.SNR_comp)
    rval = np.asarray(est.r_values)
    cnn = np.asarray(getattr(est, 'cnn_preds', np.full_like(snr, np.nan)))

    return {
        'Model': label,
        'Accepted': n_good,
        'Rejected': n_bad,
        'Total': n_total,
        'Mean SNR': np.nanmean(snr),
        'Median SNR': np.nanmedian(snr),
        'Mean r_value': np.nanmean(rval),
        'Median r_value': np.nanmedian(rval),
        'Mean CNN': np.nanmean(cnn),
        'Median CNN': np.nanmedian(cnn)
    }

# --- collect summaries for multiple CNMF runs ---
summaries = []
summaries.append(summarize_cnmf(cnm, 'contour_size'))
summaries.append(summarize_cnmf(cnm1, 'gSig=5'))
summaries.append(summarize_cnmf(cnm2, 'biblio1'))

df_summary = pd.DataFrame(summaries)
display(df_summary)

# --- distribution plots ---
def plot_distributions(models, titles):
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    metrics = ['SNR_comp', 'r_values', 'cnn_preds']
    labels = ['SNR', 'Spatial corr (r)', 'CNN']
    colors = ['tab:blue', 'tab:orange', 'tab:green', 'tab:red', 'tab:purple']

    for j, (metric, label) in enumerate(zip(metrics, labels)):
        for i, cnm in enumerate(models):
            est = cnm.estimates
            vals = np.asarray(getattr(est, metric, np.full(len(est.SNR_comp), np.nan)))
            axes[j].hist(vals, bins=40, alpha=0.4, label=titles[i], color=colors[i % len(colors)])
        axes[j].set_xlabel(label)
        axes[j].set_ylabel('Count')
        axes[j].legend()
        axes[j].grid(True)
    fig.suptitle('Quality metric distributions across models')
    plt.tight_layout()
    plt.show()

# Example usage
models = [cnm, cnm1, cnm2]
titles = ['=gSig4', 'gSig=5', 'biblio1']
plot_distributions(models, titles)


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

def summarize_cnmf(cnm, label):
    est = cnm.estimates
    n_total = est.A.shape[1]
    n_good = len(est.idx_components)
    n_bad = len(est.idx_components_bad)
    snr = np.asarray(est.SNR_comp)
    rval = np.asarray(est.r_values)
    cnn = np.asarray(getattr(est, 'cnn_preds', np.full_like(snr, np.nan)))
    return {
        'Model': label,
        'Accepted': n_good,
        'Rejected': n_bad,
        'Total': n_total,
        'Mean SNR': np.nanmean(snr),
        'Median SNR': np.nanmedian(snr),
        'Mean r_value': np.nanmean(rval),
        'Median r_value': np.nanmedian(rval),
        'Mean CNN': np.nanmean(cnn),
        'Median CNN': np.nanmedian(cnn)
    }

def plot_distributions(models, titles):
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    metrics = ['SNR_comp', 'r_values', 'cnn_preds']
    labels = ['SNR', 'Spatial corr (r)', 'CNN']
    colors = ['tab:blue', 'tab:orange', 'tab:green', 'tab:red', 'tab:purple']

    for j, (metric, label) in enumerate(zip(metrics, labels)):
        for i, cnm in enumerate(models):
            est = cnm.estimates
            vals = np.asarray(getattr(est, metric, np.full(len(est.SNR_comp), np.nan)))
            axes[j].hist(vals, bins=40, alpha=0.4, label=titles[i], color=colors[i % len(colors)])
        axes[j].set_xlabel(label)
        axes[j].set_ylabel('Count')
        axes[j].legend()
        axes[j].grid(True)
    fig.suptitle('Quality metric distributions across models')
    plt.tight_layout()
    plt.show()

def qc_scatter_with_thresholds(cnm, title):
    est = cnm.estimates
    q = cnm.params.quality
    snr = np.asarray(est.SNR_comp)
    rval = np.asarray(est.r_values)
    cnn = np.asarray(getattr(est, 'cnn_preds', np.full_like(snr, np.nan)))
    good_mask = np.isin(range(len(snr)), est.idx_components)
    bad_mask = np.isin(range(len(snr)), est.idx_components_bad)

    fig, axes = plt.subplots(1, 2, figsize=(12, 5))

    # ---- (1) r_value vs SNR
    axes[0].scatter(rval[bad_mask], snr[bad_mask], color='gray', alpha=0.4, label='Rejected')
    axes[0].scatter(rval[good_mask], snr[good_mask], color='green', alpha=0.7, label='Accepted')
    axes[0].axvline(q['rval_thr'], color='red', linestyle='--', label=f"rval_thr={q['rval_thr']}")
    axes[0].axhline(q['min_SNR'], color='orange', linestyle='--', label=f"min_SNR={q['min_SNR']}")
    axes[0].set_xlabel('Spatial corr (r)')
    axes[0].set_ylabel('SNR')
    axes[0].set_title(f'{title}: r vs SNR')
    axes[0].legend()
    axes[0].grid(True)

    # ---- (2) CNN vs SNR
    axes[1].scatter(cnn[bad_mask], snr[bad_mask], color='gray', alpha=0.4, label='Rejected')
    axes[1].scatter(cnn[good_mask], snr[good_mask], color='green', alpha=0.7, label='Accepted')
    if q.get('use_cnn', False):
        axes[1].axvline(q['min_cnn_thr'], color='red', linestyle='--', label=f"min_cnn_thr={q['min_cnn_thr']}")
    axes[1].axhline(q['min_SNR'], color='orange', linestyle='--', label=f"min_SNR={q['min_SNR']}")
    axes[1].set_xlabel('CNN prediction score')
    axes[1].set_ylabel('SNR')
    axes[1].set_title(f'{title}: CNN vs SNR')
    axes[1].legend()
    axes[1].grid(True)

    plt.tight_layout()
    plt.show()

# ---- Run summary table ----
models = [cnm, cnm1, cnm2]
titles = ['contour_size', 'gSig=5', 'biblio1']

df_summary = pd.DataFrame([summarize_cnmf(m, t) for m, t in zip(models, titles)])
display(df_summary)

# ---- Plot metric distributions across models ----
plot_distributions(models, titles)

# ---- Plot per-model QC scatter with thresholds ----
for cnm, title in zip(models, titles):
    qc_scatter_with_thresholds(cnm, title)
